In [0]:
%run ./03_Scenario_Simulation

root
 |-- date: date (nullable = true)
 |-- store_id: long (nullable = true)
 |-- store_region: string (nullable = true)
 |-- sku_id: long (nullable = true)
 |-- category: string (nullable = true)
 |-- units_sold: long (nullable = true)
 |-- revenue: double (nullable = true)
 |-- promo_flag: long (nullable = true)
 |-- promo_type: string (nullable = true)
 |-- price: double (nullable = true)
 |-- inventory_level: long (nullable = true)
 |-- store_size: string (nullable = true)
 |-- holiday_flag: long (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- revenue_per_unit: double (nullable = true)
 |-- is_promo: boolean (nullable = true)



In [0]:
simulate_supply_shortage(15)

{'simulation_type': 'Supply Shortage',
 'shortage_percent': 15,
 'baseline_revenue': 339347.93,
 'projected_revenue': 288445.74,
 'revenue_impact_percent': -15.0,
 'baseline_units': 60573.0,
 'projected_units': 51487.05}

In [0]:
%pip install langchain-openai

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import requests
import json

workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

ENDPOINT_NAME = "databricks-gpt-oss-20b"  

In [0]:
def call_llm(prompt):

    url = f"https://{workspace_url}/serving-endpoints/{ENDPOINT_NAME}/invocations"

    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }

    payload = {
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }

    response = requests.post(url, headers=headers, json=payload)

    if response.status_code != 200:
        return f"Error: {response.status_code}, {response.text}"

    data = response.json()

    if "choices" in data:
        content_blocks = data["choices"][0]["message"]["content"]

        if isinstance(content_blocks, list):
            for block in content_blocks:
                if block.get("type") == "text":
                    return block.get("text", "")
        else:
            return content_blocks

    return "No valid text returned from model."


In [0]:
print(call_llm("Explain retail supply shortage impact in 3 lines."))

Retail supply shortages push prices higher, squeezing consumer purchasing power and eroding brand loyalty. They strain distributor networks, forcing retailers to cut back inventory, delay promotions, and reduce in‑store footfall. Over time, prolonged shortages can disrupt brand reputation, depress sales, and increase operational costs as companies scramble to secure alternative sources.


In [0]:
def tool_supply_shortage(percent: int):
    """
    Simulates supply shortage impact on revenue.
    Input: percent reduction in inventory.
    Output: dictionary with revenue impact metrics.
    """
    return simulate_supply_shortage(percent)


def tool_price_increase(percent: int):
    """
    Simulates price increase impact using elasticity.
    """
    return simulate_price_increase(percent)


def tool_promo_uplift(percent: int):
    """
    Simulates demand uplift for promotional items.
    """
    return simulate_promo_uplift(percent)


def tool_combined(price_percent: int, promo_percent: int):
    """
    Simulates combined price increase and promo uplift.
    """
    return simulate_combined_strategy(price_percent, promo_percent)


In [0]:
TOOLS = {
    "supply_shortage": tool_supply_shortage,
    "price_increase": tool_price_increase,
    "promo_uplift": tool_promo_uplift,
    "combined_strategy": tool_combined
}

In [0]:
import re

def select_simulation_tool(user_input):

    text = user_input.lower()
    numbers = re.findall(r"\d+", text)

    default_percent = 10

    trigger_words = ["simulate", "what if", "increase", "reduce", "run", "model"]

    if any(word in text for word in trigger_words):

        if "supply" in text or "shortage" in text:
            percent = int(numbers[0]) if numbers else default_percent
            return "supply_shortage", percent

        elif "price" in text:
            percent = int(numbers[0]) if numbers else default_percent
            return "price_increase", percent

        elif "promo" in text or "promotion" in text:
            percent = int(numbers[0]) if numbers else default_percent
            return "promo_uplift", percent

        elif "combined" in text:
            if len(numbers) >= 2:
                return "combined_strategy", (int(numbers[0]), int(numbers[1]))
            else:
                return "combined_strategy", (5, 10)

    return None, None


In [0]:
def agent_executor(user_input):

    tool_name, param = select_simulation_tool(user_input)

    if tool_name not in TOOLS:
        return "No valid tool detected."

    if tool_name == "combined_strategy":
        result = TOOLS[tool_name](param[0], param[1])
    else:
        result = TOOLS[tool_name](param)

    prompt = f"""
    Business Question:
    {user_input}

    Simulation Output:
    {result}

    Provide a strategic executive memo.
    """

    return call_llm(prompt)


In [0]:
print(agent_executor("Simulate supply shortage of 15 percent."))

**To:** Chief Executive Officer, Board of Directors, and Executive Leadership Team  
**From:** [Analyst/Chief Strategy Officer]  
**Date:** 17 Feb 2026  
**Subject:** Executive Summary – 15 % Supply‑Shortage Impact Simulation

---

### 1. Purpose

This memo summarizes the key findings from the most recent supply‑shortage simulation, whereby a 15 % reduction in critical raw‑material availability was modeled against our current operating baseline. The objective is to provide a clear, data‑driven view of the projected revenue decline, its underlying drivers, and actionable recommendations to mitigate risk.

---

### 2. Simulation Snapshot

| Metric | Baseline | Projected (15 % shortage) | Impact |
|--------|----------|---------------------------|--------|
| Units Sold | 60,573 | 51,487 | ↓ 15 % |
| Unit Revenue (USD) | 5,597.0 | 5,597.0 (unchanged) | — |
| Total Revenue | **339,347.93** | **288,445.74** | **↓ 15 %** |
| Revenue Impact | — | — | **‑15 %** |

*Key Assumption: Unit price rem

In [0]:
print(agent_executor("What happens if we increase price by 8 percent?"))

**To:** Executive Leadership Team  
**From:** Senior Strategy & Analytics Office  
**Date:** 17 Feb 2026  
**Subject:** Impact of an 8 % Premium Increase – Strategic Briefing  

---

### 1. Executive Summary  
A one‑time 8 % increase in the spot price of our flagship product, implemented across all SKU‑bundles, projects a **revenue lift of 3.68 %** (≈ US $12,488) for FY 2026‑27. Units sold would decline by **≈ 4 %** (≈ 2,423 units). The net effect is a modest positive upside to top‑line revenue without a proportionate drop in sales volume.

---

### 2. Simulation Snapshot  

| Metric | Baseline | Projected (8 % ↑) | Δ | % Change |
|--------|----------|-------------------|---|----------|
| Units Sold | 60,573 | 58,150 | –2,423 | –4.0 % |
| Unit Price | – | +8 % | +8 % | +8 % |
| Revenue | $339,347.93 | $351,835.94 | +$12,488.01 | +3.68 % |

> **Note:** Costs are held constant in this model; margin impact is therefore identical to the revenue effect unless cost structures shift.

---

##

In [0]:
print(agent_executor("Run promo uplift of 20 percent."))

---

**To:**  Senior Executive Team  
**From:**  [Your Name], Director of Growth & Analytics  
**Date:**  17 Feb 2026  
**Subject:**  Strategic Outlook on 20 % Promo Uplift Initiative  

---

### Executive Summary  

A rigorous lift‑test simulation confirms that a **20 % promotional uplift** will translate into an **additional $20 145 in revenue** (≈5.94 % increase) and an extra **4 199 units sold** over the current baseline. While modest on a portfolio‑wide scale, the lift provides a solid foundation to refine our go‑to‑market strategies, optimize channel mix, and safeguard margin.  

Key take‑aways:

| Metric | Baseline | Projected | Δ | % Change |
|--------|----------|-----------|---|----------|
| Revenue | $339 348 | $359 492 | +$20 145 | +5.94 % |
| Units | 60 573 | 64 772 | +4 199 | +6.91 % |
| Promo Uplift | — | — | — | 20 % (by design) |

**Recommendation** – Launch the promo in high‑margin, high‑velocity segments (A‑tier SKUs in digital marketplaces) with an aggressive digital